### Key Steps

#### 1. Read data from the CSV
#### 2. Predicting zone temperature and compressor power using RNN with LSTM architecture
#### 3. Perform hyperparameter tuning usig optuna
#### 4. Implement a RL algorithm (PPO) to decide optimal system state (on/off) and zone set-points to minimize peak demand over a look ahead window

#### Step 0: Import packages 

In [ ]:
import pandas as pd
import tensorflow as tf
import sklearn as sk
import numpy as np  
import matplotlib.pyplot as plt
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import optuna


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_absolute_error, r2_score 
from sklearn.preprocessing import StandardScaler



# Predict zone temperature and power using RNN (with LSTM). Input features are historical zone temp, zone temp set-points, ambient temp and system state (on/off)

##### 1 Data preparation - General

In [2]:
# Read appropriate data files
evap_raw_data_df = pd.read_csv('evap_raw_data.csv')
com_and_other_raw_data_df = pd.read_csv('comp_and_other_raw_data.csv')


# Combine the two dataframes and remove any duplicate columns and remove duplicate columns
combined_df = pd.merge(evap_raw_data_df, com_and_other_raw_data_df, on = 'DateTime', how = 'inner')

# set index
combined_df['DateTime'] = pd.to_datetime(combined_df['DateTime'])
combined_df.set_index('DateTime', inplace = True)

#### 2  Data preparation - refrigeration power and evap temp modeling

In [ ]:

# Extract evap names 
evap_names_raw = []
for x in combined_df.columns:
    first_part = x.split(' - ')[0]
    if 'CG ' in first_part:
        evap_names_raw.append(first_part.split(' ')[1])
evap_names = list(set(evap_names_raw))

#dictionary of name mappings
evap_points_name_dict = {}
for evap in evap_names:
    for x in combined_df.columns:
        if evap in x and ('SP' in x or 'Setpoint' in x):
            evap_points_name_dict[x] = evap+'_temp_setpoint'
        if evap in x and 'Actual Temp' in x:
            evap_points_name_dict[x] = evap+'_temp'
            
            
# print those evaps which have missing points
evaps_missing_setpoints = []
evaps_missing_temps = []
for evap in evap_names:
    if evap+'_temp_setpoint' not in evap_points_name_dict.values():
        evaps_missing_setpoints.append(evap)
    if evap+'_temp' not in evap_points_name_dict.values():
        evaps_missing_temps.append(evap)
print('Evaps missing setpoints:', evaps_missing_setpoints)
print('Evaps missing temps:', evaps_missing_temps)

# create a new dataframe with only the columns of interest
reformatted_df = combined_df[evap_points_name_dict.keys()]
reformatted_df.rename(columns = evap_points_name_dict, inplace = True)

# fill in missing setpoints and temps
# map of evaps with missing temps and those which have temps
evaps_map_for_missing_temps = [('H06','H05'), ('G06','G05'), ('G04', 'G03'), ('H08', 'H07'), ('H04', 'H03'), ('H10', 'H09'), ('G02', 'G01'), ('G08', 'G07'), ('H02', 'H01') ]
for pair in evaps_map_for_missing_temps:
    reformatted_df[pair[0]+'_temp'] = reformatted_df[pair[1]+'_temp']
    
# add exogenous variables
reformatted_df['dry_bulb_temp'] = combined_df['Condenser Sequencer - Air Temperature']
reformatted_df['wet_bulb_temp'] = combined_df['Condenser Sequencer - Wetbulb Temperature']

# compute total compressor amps and total power
# find all columns with amps in them
amps_columns = [x for x in combined_df.columns if 'Amps' in x]
reformatted_df['total_compressor_amps'] = combined_df[amps_columns].sum(axis = 1)
reformatted_df['total_compressor_power'] = reformatted_df['total_compressor_amps'] * 480*(3**0.5)*0.93/1000
reformatted_df.drop(columns = 'total_compressor_amps', inplace = True)

# Add another feature called system on or off based on whether the total compressor power is greater than 25
reformatted_df['system_on'] = reformatted_df['total_compressor_power'] > 25
reformatted_df['system_on'] = reformatted_df['system_on'].astype(int)

# select features relevant to the problem
features = []
for evap in evap_names:
    features.append(evap+'_temp')
    features.append(evap+'_temp_setpoint')
features.append('dry_bulb_temp')
features.append('system_on')
features.append('total_compressor_power')

# create a dataframe for this learning task
power_modeling_dataset = reformatted_df[features]

# drop rows with NaN values
power_modeling_dataset.dropna(inplace = True)

#function to create sequences of data
def create_sequences(data, seq_length, time_indices, input_feature_indices, output_feature_indices):
    sequences = []
    targets = []
    target_time_indices = []
    for i in range(len(data) - seq_length):
        seq = data[i:i+seq_length, input_feature_indices]
        target = data[i+seq_length, output_feature_indices]
        target_time_index = time_indices[i+seq_length]
        sequences.append(seq)
        targets.append(target)
        target_time_indices.append(target_time_index)
    return np.array(sequences), np.array(targets), target_time_indices

# do test train split
seq_length = 1 # change this to increase the sequence length
train_data, test_data = train_test_split(power_modeling_dataset, test_size = 0.2, shuffle = False)

# scale the data such that each column has its own scaler 
scalers = {}
train_data_scaled = train_data.copy()
test_data_scaled = test_data.copy()
for col in train_data.columns:
    scaler = StandardScaler()
    train_data_scaled[col] = scaler.fit_transform(train_data[[col]])
    test_data_scaled[col] = scaler.transform(test_data[[col]])
    scalers[col] = scaler    

# # scale the data using a single scaler
# scaler = StandardScaler()
# train_data_scaled = scaler.fit_transform(train_data)
# test_data_scaled = scaler.transform(test_data)

# convert to numpy arrays
train_data_scaled = train_data_scaled.to_numpy()
test_data_scaled = test_data_scaled.to_numpy()

# get column numbers of the features
input_feature_indices = []
for evap in evap_names:
    input_feature_indices.append(power_modeling_dataset.columns.get_loc(evap+'_temp'))
    input_feature_indices.append(power_modeling_dataset.columns.get_loc(evap+'_temp_setpoint'))
input_feature_indices.append(power_modeling_dataset.columns.get_loc('dry_bulb_temp'))
input_feature_indices.append(power_modeling_dataset.columns.get_loc('system_on'))

output_feature_indices = []
dict_of_output_feature_indices = {}
idx = 0
for evap in evap_names:
    output_feature_indices.append(power_modeling_dataset.columns.get_loc(evap+'_temp'))
    dict_of_output_feature_indices[evap] = idx
    idx += 1
output_feature_indices.append(power_modeling_dataset.columns.get_loc('total_compressor_power'))
dict_of_output_feature_indices['total_compressor_power'] = idx

# create sequences
x_train_seq, y_train, training_indices = create_sequences(train_data_scaled, seq_length, train_data.index, input_feature_indices, output_feature_indices)
x_test_seq, y_test, testing_indices = create_sequences(test_data_scaled, seq_length, test_data.index, input_feature_indices, output_feature_indices)

#convert to pytorch tensors
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x_train_seq_tensor = torch.tensor(x_train_seq, dtype = torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype = torch.float32).to(device)
x_test_seq_tensor = torch.tensor(x_test_seq, dtype = torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype = torch.float32).to(device)




#### 3 Define a basic LSTM model 

In [4]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, dropout):
        # initialize the model by inheriting from nn.Module 
        super().__init__()
        
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        
        #define the LSTM ensemble of layers 
        self.lstm_layer = nn.LSTM(input_size, hidden_size, num_layers, batch_first = True, dropout = dropout) 
        
        #define the output layer (a fully connected ANN)
        self.output_layer = nn.Linear(hidden_size, output_size)
        
    # define function for forward pass. Note this is done per batch and so number of batches does not matter 
    def forward(self, x):
        # first we initialize the hiden and cell states. Initialization is necessary. 
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device) #note that there are num_layers*hidden_size params (hidden states) for each sequence in a batch. So the total number of hidden states is num_layers*hidden_size*num_of_sequences per batch aka batch size
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device) # same dimensionality comment as above
        
        out, _ = self.lstm_layer(x, (h0, c0)) #In this step, the batched data in x is taken and passed through the LSTM layers, per time-step. For each sequence in the batch it generates the output feature per time-step. Hence output will be a tensor of size batch_size*sequence_length*hidden_size
        
        #next, we take the outputs of the last time-step (in each sequence) and pass it though the output layer. 
        out = self.output_layer(out[:,-1, :]).to(x.device)      
        
        return out
        
            # Might be easier to think of a LSTM as a regular NN except that each element in a batch is a sequence (unfolded in time)
        

#### 4 Create an instance of the LSTM model using hyperparameters, train the model, evaluate it and compute test loss. All of this wrapped up within hyperparameter tuning

In [ ]:

# hyperparameter tuning using optuna

def objective(trial):
    # define parameters to optimize
    hidden_size = trial.suggest_int('hidden_size',16, 128)
    num_layers = trial.suggest_int('num_layers', 1, 8)
    dropout_prob = trial.suggest_float('dropout_prob', 0.0, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2)
    epochs = trial.suggest_int('epochs', 100, 300)
    
    #initialize model
    input_size = x_train_seq_tensor.shape[2]
    output_size = y_train_tensor.shape[1] 
    model = LSTMModel(input_size, hidden_size, output_size, num_layers, dropout_prob).to(device)
    
    # Define the loss function
    loss_function = nn.MSELoss()
    
    # Define the optimizer to use and learning rate
    optimizer = optim.Adam(model.parameters(), lr = learning_rate)
    
    #train the model
    for epoch in range(epochs):
        model.train() # set model to training mode
        optimizer.zero_grad() # zero the gradients otherwise they will accumulate
        
        #forward pass
        outputs = model(x_train_seq_tensor)
        
        #calculate loss
        loss = loss_function(outputs, y_train_tensor)
        
        #backward pass
        loss.backward()
        
        #update the weights
        optimizer.step()
        
    # evaluate the model
    model.eval() # set model to evaluation mode
    with torch.no_grad():
        y_pred = model(x_test_seq_tensor)
        test_loss = loss_function(y_pred, y_test_tensor)
            
    return test_loss.item()

# create a study object and optimize
study = optuna.create_study(direction = 'minimize')
study.optimize(objective, n_trials = 100)

# get the best parameters
best_params = study.best_trial.params




        

    


In [ ]:
best_params

#### 5 Create an instance of the model (either using manually specified or optimized hyperparams) and also to train and evaluate it

In [ ]:
# PLEASE COMMENT ONE OF THE TWO OPTIONS BELOW 
   
# # Option 1: Manually specified Hyperparameters    
# hidden_size = 32
# num_layers = 2
# dropout_prob = 0.2
# learning_rate = 0.005
# num_epochs = 200

# Option 2: Use the best parameters from the hyperparameter tuning
hidden_size = best_params['hidden_size']
num_layers = best_params['num_layers']
dropout_prob = best_params['dropout_prob']
learning_rate = best_params['learning_rate']
num_epochs = best_params['epochs']

    
#initialize model
input_size = x_train_seq_tensor.shape[2]
output_size = y_train_tensor.shape[1] 
model = LSTMModel(input_size, hidden_size, output_size, num_layers, dropout_prob).to(device)
    
# Define the loss function
loss_function = nn.MSELoss()
    
# Define the optimizer to use and learning rate
optimizer = optim.Adam(model.parameters(), lr = learning_rate)
    

#Training loop
epochs = []
loss_values = []
for epoch in range(num_epochs):
    model.train() # set model to training mode
    optimizer.zero_grad() # zero the gradients otherwise they will accumulate
    
    #forward pass
    outputs = model(x_train_seq_tensor)
    
    #calculate loss
    loss = loss_function(outputs, y_train_tensor)
    
    #backward pass
    loss.backward()
    
    #update the weights
    optimizer.step()
    
    #print and log the loss
    if epoch %10 ==0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')
    epochs.append(epoch)
    loss_values.append(loss.item())

# plot the loss values using matplotlib
plt.plot(epochs, loss_values)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

# evaluate the model
model.eval() # set model to evaluation mode
with torch.no_grad():
    y_pred = model(x_test_seq_tensor)
    test_loss = loss_function(y_pred, y_test_tensor)
    print(f'Test loss: {test_loss.item()}')
    
# plot the actual vs predicted values for both training and testing sets
y_training_pred = model(x_train_seq_tensor)
y_testing_pred = model(x_test_seq_tensor)


        

#### 7 plot the results

In [ ]:
import plotly.graph_objects as go

# convert to numpy arrays
y_training_pred_numpy= y_training_pred.detach().cpu().numpy()
y_train_numpy = y_train_tensor.detach().cpu().numpy()
y_testing_pred_numpy = y_testing_pred.detach().cpu().numpy()
y_test_numpy = y_test_tensor.detach().cpu().numpy()

# For a given evap, extract the actual and predicted values (unscaled) for the evap temp
evap_to_model = 'A01'
evap_temp_index = dict_of_output_feature_indices[evap_to_model]
y_training_pred_evap = y_training_pred_numpy[:, evap_temp_index]
y_train_evap = y_train_numpy[:, evap_temp_index]
y_testing_pred_evap = y_testing_pred_numpy[:, evap_temp_index]
y_test_evap = y_test_numpy[:, evap_temp_index]

# unscale the above actual and predicted values
y_training_pred_evap_unscaled = scalers[evap_to_model+'_temp'].inverse_transform(y_training_pred_evap.reshape(-1,1)).flatten()
y_train_evap_unscaled = scalers[evap_to_model+'_temp'].inverse_transform(y_train_evap.reshape(-1,1)).flatten()
y_testing_pred_evap_unscaled = scalers[evap_to_model+'_temp'].inverse_transform(y_testing_pred_evap.reshape(-1,1)).flatten()
y_test_evap_unscaled = scalers[evap_to_model+'_temp'].inverse_transform(y_test_evap.reshape(-1,1)).flatten()

#extract the actual and predicted values for the power
power_index = dict_of_output_feature_indices['total_compressor_power']
y_training_pred_power = y_training_pred_numpy[:, power_index]
y_train_power = y_train_numpy[:, power_index]
y_testing_pred_power = y_testing_pred_numpy[:, power_index]
y_test_power = y_test_numpy[:, power_index]

# unscale the above actual and predicted values
y_training_pred_power_unscaled = scalers['total_compressor_power'].inverse_transform(y_training_pred_power.reshape(-1,1)).flatten()
y_train_power_unscaled = scalers['total_compressor_power'].inverse_transform(y_train_power.reshape(-1,1)).flatten()
y_testing_pred_power_unscaled = scalers['total_compressor_power'].inverse_transform(y_testing_pred_power.reshape(-1,1)).flatten()
y_test_power_unscaled = scalers['total_compressor_power'].inverse_transform(y_test_power.reshape(-1,1)).flatten()



# plot actual versus predicted evap temp for the training set using Plotly
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=training_indices, y=y_train_evap_unscaled, mode='lines', name='Actual'))  # Actual values    
fig1.add_trace(go.Scatter(x=training_indices, y=y_training_pred_evap_unscaled, mode='lines', name='Predicted'))  # Predicted values
fig1.update_layout(title=f'Actual vs Predicted values for training set for evaporator {evap_to_model}', xaxis_title='Time', yaxis_title='Temperature')
fig1.update_layout(template='plotly_dark')
fig1.show()

# plot actual versus predicted evap temp for the testing set using Plotly
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=testing_indices, y=y_test_evap_unscaled, mode='lines', name='Actual'))  # Actual values
fig2.add_trace(go.Scatter(x=testing_indices, y=y_testing_pred_evap_unscaled, mode='lines', name='Predicted'))  # Predicted values
fig2.update_layout(title=f'Actual vs Predicted values for testing set for evaporator {evap_to_model}', xaxis_title='Time', yaxis_title='Temperature')
fig2.update_layout(template='plotly_dark')
fig2.show()

# plot actual versus predicted power for the training set using Plotly
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=training_indices, y=y_train_power_unscaled, mode='lines', name='Actual'))  # Actual values
fig3.add_trace(go.Scatter(x=training_indices, y=y_training_pred_power_unscaled, mode='lines', name='Predicted'))  # Predicted values
fig3.update_layout(title='Actual vs Predicted values for training set for power modeling', xaxis_title='Time', yaxis_title='Power')
fig3.update_layout(template='plotly_dark')
fig3.show()

# plot actual versus predicted power for the testing set using Plotly
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=testing_indices, y=y_test_power_unscaled, mode='lines', name='Actual'))  # Actual values
fig4.add_trace(go.Scatter(x=testing_indices, y=y_testing_pred_power_unscaled, mode='lines', name='Predicted'))  # Predicted values
fig4.update_layout(title='Actual vs Predicted values for testing set for power modeling', xaxis_title='Time', yaxis_title='Power')
fig4.update_layout(template='plotly_dark')
fig4.show()



# Use RL to minimize peak demand and keep evap temps under bounds 

In [ ]:
# install the following packages: gym, stable-baselines3
!pip install gym
!pip install stable-baselines3


#### 8. Define the RL environment 

In [9]:
# Use RL (PPO) to keep evap setpoints within a certain range while minimizing power peak demand over a look ahead window. The decision variables are evap set-point temps, and system on/off state. Assume that the ambient temp forecast over the look ahead window is known.

# Define the environment
import gym
from gym import spaces 
import numpy as np

class RefrigerationControlEnv(gym.Env):
    def __init__(self, lstm_model, scalers, initial_state, forecast_ambient_temp, evap_temp_bounds, look_ahead_window, evap_names, evap_setpoint_bounds):
        super().__init__()
        
        # model and scaler for predictions 
        self.lstm_model = lstm_model
        self.scalers = scalers 
        
        # prediction related variables
        self.look_ahead_window = look_ahead_window
        self.forecast_ambient_temp = forecast_ambient_temp
        
        # state related variables (states are evap temps and compressor power)
        self.initial_state = initial_state
        
        # bounds
        self.evap_temp_bounds = evap_temp_bounds #note evap_temp_bounds is a dictionary with evap names as keys and a tuple of min and max temps as values
        
        # action space:
        num_evaps = len(evap_names)
        #self.action_space = spaces.Box(low =-1, high = 1, shape = (num_evaps+1,), dtype = np.float32) # the last element in the action space is the system on/off state
        
        # constrain the action space such that the evap setpoints are within the bounds and the system on/off state is binary
        self.action_space = spaces.Box(low = np.array([evap_setpoint_bounds[evap][0] for evap in evap_names] + [0]), high = np.array([evap_setpoint_bounds[evap][1] for evap in evap_names] + [1]), dtype = np.float32)       
              
        
        # observation space: note that the observation space is the state space
        self.observation_space = spaces.Box(low = -np.inf, high = np.inf, shape = (num_evaps+1,), dtype = np.float32)
        
        # track the power peak
        self.max_compressor_power = 0
        
        # evap names
        self.evap_names = evap_names
        
        
    # define the step function
    def step(self, action):
        # extract the evap set-points and system on/off state from the action. 
        evap_setpoints = action[:-1]
        system_on = action[-1]
        system_on = 1 if system_on > 0.5 else 0
        
        # update the state/outputs and time step
        self.state = self.update_state(evap_setpoints, system_on)
        evap_temps = self.state[:-1]
        compressor_power = self.state[-1]
        self.max_compressor_power = max(self.max_compressor_power, compressor_power)
        self.current_time_step += 1
        
        # calculate the reward
        peak_demand_reward = -self.max_compressor_power
        temp_violation_reward = self.calculate_temp_violation_reward(evap_temps)
        current_reward = peak_demand_reward + temp_violation_reward
        
        # check if episode is done
        done=self.check_done()
        return self.state, current_reward, done, {}
    
    # define the other functions
    
    #first define a reset function that resets the environment to the initial state for a new episode
    def reset(self):
        self.state = self.initial_state
        self.max_compressor_power = 0
        self.current_time_step = 0
        return self.state
    
    # next define the update_state function that updates the state at time time t given the evap setpoints and system on/off state at time t. (We assume that the LSTM model has sequence length of 1 for simplicity)
    def update_state(self, evap_setpoints, system_on):
        # first update the state by invoking the LSTM model to get the forecasted evap temps and compressor power
        self.state = self.get_forecasted_state(evap_setpoints, system_on)
        return self.state
        
    # Now define the get_forecasted_state function that uses the LSTM model to get the forecasted evap temps and compressor power
    def get_forecasted_state(self, evap_setpoints, system_on):
        # extract the evap temps from the current state and scale them
        evap_temps = self.state[:-1]
        evap_temps_scaled = self.scale_evap_temps(evap_temps)
        
        # Next scale the evap setpoints
        evap_setpoints_scaled = self.scale_evap_setpoints(evap_setpoints)
        
        # next scale the forecasted ambient temp
        forecast_ambient_temp_scaled = self.scalers['dry_bulb_temp'].transform(np.array([self.forecast_ambient_temp[self.current_time_step]]).reshape(-1, 1))[0, 0]

        
        # next scale the system on/off state        
        system_on_scaled = self.scalers['system_on'].transform(np.array([system_on]).reshape(-1, 1))[0, 0]
        
        # combine the above into a single input tensor following the order of the input features. Recall that the input features are in the order: evap temp for evap 1, evap temp setpoint for evap 1, evap temp for evap 2, evap temp setpoint for evap 2, ..., dry bulb temp, system on/off state
        # for this example, we assume that the evap temps and setpoints are in the same order as the initial state
        input_tensor = []
        for i in range(len(evap_temps)):
            input_tensor.append(evap_temps_scaled[i])
            input_tensor.append(evap_setpoints_scaled[i])
        input_tensor.append(forecast_ambient_temp_scaled)
        input_tensor.append(system_on_scaled)              
    
        
        # convert to a tensor
        input_tensor = torch.tensor(input_tensor, dtype = torch.float32).to(device)
        
        #reshape it to be a tensor of the shape expected by the LSTM model
        input_tensor = input_tensor.reshape(1, 1, -1)
        
         
        # get the forecasted evap temps and compressor power
        with torch.no_grad():
            forecasted_outputs = self.lstm_model(input_tensor)
            forecasted_evap_temps = forecasted_outputs[:, :-1]
            forecasted_compressor_power = forecasted_outputs[:, -1]
        
        # unscale the forecasted evap temps and compressor power
        forecasted_evap_temps_unscaled = self.unscale_evap_temps(forecasted_evap_temps.detach().cpu().numpy().flatten())
        forecasted_compressor_power_unscaled = self.scalers['total_compressor_power'].inverse_transform(forecasted_compressor_power.detach().cpu().numpy().reshape(-1, 1))[0, 0]

        # combine the forecasted evap temps and compressor power into a single state tensor
        # Ensure both arrays are 1D for concatenation
        forecasted_compressor_power_unscaled = np.array([forecasted_compressor_power_unscaled])  # Convert scalar to 1D array
        state = np.concatenate([forecasted_evap_temps_unscaled, forecasted_compressor_power_unscaled], axis=0)

        
        return state
    
    # Define functions to scale and unscale the evap temps and setpoints
    def scale_evap_temps(self, evap_temps):
        evap_temps_scaled = []
        evap_names = self.evap_names
        for evap, i in zip(evap_names, range(len(evap_temps))):
            evap_temp_scaler = self.scalers[evap + '_temp']
            # Wrap the scalar value in a 2D array and extract the result
            scaled_value = evap_temp_scaler.transform([[evap_temps[i]]])[0, 0]
            evap_temps_scaled.append(scaled_value)
        return evap_temps_scaled

    def scale_evap_setpoints(self, evap_setpoints):
        evap_setpoints_scaled = []
        evap_names = self.evap_names
        for evap, i in zip(evap_names, range(len(evap_setpoints))):
            evap_temp_setpoint_scaler = self.scalers[evap + '_temp_setpoint']
            # Wrap the scalar value in a 2D array and extract the result
            scaled_value = evap_temp_setpoint_scaler.transform([[evap_setpoints[i]]])[0, 0]
            evap_setpoints_scaled.append(scaled_value)
        return evap_setpoints_scaled

    def unscale_evap_temps(self, forecasted_evap_temps):
        forecasted_evap_temps_unscaled = []
        evap_names = self.evap_names
        for evap, i in zip(evap_names, range(len(forecasted_evap_temps))):
            evap_temp_scaler = self.scalers[evap + '_temp']
            # Wrap the scalar value in a 2D array and extract the result
            unscaled_value = evap_temp_scaler.inverse_transform([[forecasted_evap_temps[i]]])[0, 0]
            forecasted_evap_temps_unscaled.append(unscaled_value)
        return forecasted_evap_temps_unscaled

    
    def calculate_temp_violation_reward(self, evap_temps):
        temp_violation_reward = 0
        evap_names = self.evap_names
        for evap, i in zip(evap_names, range(len(evap_temps))):
            if evap_temps[i] < self.evap_temp_bounds[evap][0] or evap_temps[i] > self.evap_temp_bounds[evap][1]:
                temp_violation_reward -= 1000
        return temp_violation_reward
    
    def check_done(self):
        done = False
        if self.current_time_step == self.look_ahead_window:
            done = True
        return done

        
        

#### 9. Train the agent with PPO

In [ ]:
# Train the agent using PPO

import stable_baselines3
from stable_baselines3 import PPO

#lstm_model, scalers, initial_state, forecast_ambient_temp, evap_temp_bounds, look_ahead_window, evap_names):

# Create the attributes for the environment
initial_evap_temps = []
for evap in evap_names:
    initial_evap_temps.append(test_data[evap+'_temp'].iloc[0])
initial_state = np.concatenate([initial_evap_temps, [test_data['total_compressor_power'].iloc[0]]], axis = 0)

forecast_ambient_temp = test_data['dry_bulb_temp'].to_numpy()

evap_temp_bounds = {}
for evap in evap_names:
    evap_temp_bounds[evap] = (power_modeling_dataset[evap+'_temp'].min(), power_modeling_dataset[evap+'_temp'].max())
    
look_ahead_window = len(test_data)

evap_setpoint_bounds = {}
for evap in evap_names:
    evap_setpoint_bounds[evap] = (power_modeling_dataset[evap+'_temp_setpoint'].min()-2, power_modeling_dataset[evap+'_temp_setpoint'].max()+2) # add a buffer of 2 degrees to the min and max setpoints

# create an instance of the environment
env = RefrigerationControlEnv(lstm_model=model, scalers=scalers, initial_state=initial_state, forecast_ambient_temp=forecast_ambient_temp, evap_temp_bounds=evap_temp_bounds, look_ahead_window=look_ahead_window, evap_names=evap_names, evap_setpoint_bounds=evap_setpoint_bounds)


# now create the PPO agent
agent = PPO('MlpPolicy', env, verbose = 1)

# train the agent
agent.learn(total_timesteps = 10000)



#### 10. Evaluate the optimal policy 

In [ ]:
# Evaluate the optimal policy. Its akin to simulating a full episode using the learned policy

obs = env.reset()
optimal_evap_setpoints = []
optimal_system_on = []
optimal_evap_temps = []
optimal_compressor_power = []

# Loop over the look ahead window
for i in range(look_ahead_window):
    action, _states = agent.predict(obs, deterministic = True) # note that the _states variable indicates the hidden states of the LSTM model and is not used here
    obs, rewards, done, info = env.step(action)
    
    # extract the optimal evap setpoints over the look ahead window
    optimal_evap_setpoints.append(action[:-1])
    
    #extract the optimal system on/off state over the look ahead window
    optimal_system_on.append(action[-1])
    
    # extract the resulting evap temps and compressor power over the look ahead window
    optimal_evap_temps.append(obs[:-1])    
    optimal_compressor_power.append(obs[-1])
    
        
    if done:
        break
    
    
# plot the compressor power over the look ahead window using plotly using the timestamps
timestamps = test_data.index
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=timestamps, y=optimal_compressor_power, mode='lines', name='Optimal Compressor Power'))  # Compressor Power
# add trace for compressor power in the test data
fig5.add_trace(go.Scatter(x=timestamps, y=test_data['total_compressor_power'], mode='lines', name='Actual Compressor Power'))  # Compressor Power
fig5.update_layout(title='Optimal Compressor Power over look ahead window', xaxis_title='Time', yaxis_title='Power')
fig5.update_layout(template='plotly_dark')
fig5.show()

# plot the evap temps and set-points over the look ahead window using plotly using the timestamps for a given evap
evap_to_model = 'A01'
evap_temp_index = evap_names.index(evap_to_model)
optimal_evap_temps_evap = [x[evap_temp_index] for x in optimal_evap_temps]
optimal_evap_setpoints_evap = [x[evap_temp_index] for x in optimal_evap_setpoints]
fig6 = go.Figure()
fig6.add_trace(go.Scatter(x=timestamps, y=optimal_evap_temps_evap, mode='lines', name='Optimal Evap Temp'))  # Evap Temp
fig6.add_trace(go.Scatter(x=timestamps, y=optimal_evap_setpoints_evap, mode='lines', name='Optimal Evap Setpoint'))  # Evap Setpoint
# add trace for evap temp and setpoint in the test data
fig6.add_trace(go.Scatter(x=timestamps, y=test_data[evap_to_model+'_temp'], mode='lines', name='Actual Evap Temp'))  # Evap Temp
fig6.add_trace(go.Scatter(x=timestamps, y=test_data[evap_to_model+'_temp_setpoint'], mode='lines', name='Actual Evap Setpoint'))  # Evap Setpoint
fig6.update_layout(title=f'Optimal Evap Temp and Setpoint over look ahead window for evaporator {evap_to_model}', xaxis_title='Time', yaxis_title='Temperature')
fig6.update_layout(template='plotly_dark')
fig6.show()

#plot the system on/off state over the look ahead window using plotly using the timestamps
optimal_system_on = np.array(optimal_system_on)
fig7 = go.Figure()
fig7.add_trace(go.Scatter(x=timestamps, y=optimal_system_on, mode='lines', name='Optimal System On/Off'))  # System On/Off
# add trace for system on/off in the test data
fig7.add_trace(go.Scatter(x=timestamps, y=test_data['system_on'], mode='lines', name='Actual System On/Off'))  # System On/Off
fig7.update_layout(title='Optimal System On/Off over look ahead window', xaxis_title='Time', yaxis_title='System On/Off')
fig7.update_layout(template='plotly_dark')
fig7.show()

# print the evap setpoint bounds for the given evap
print('Evap setpoint bounds:')
print(evap_setpoint_bounds[evap_to_model])

#print the evap temp bounds for the given evap
print('Evap temp bounds:')
print(evap_temp_bounds[evap_to_model])









    

